# Boulder recommendation  
## Similarity calculation taking into account ascents and grade 
Ascents: Jaccard similarity  
Grades: Cosine similarity  

## SQLAlchemy session creation

In [1]:
import numpy as np
import pandas as pd


from sqlalchemy.orm import Session
from sqlalchemy import create_engine

DB_URL = "sqlite:///../../matchit.db"

engine = create_engine(DB_URL, echo=False)

session = Session(engine)

In [2]:
import sys

sys.path.append("../../")

## Similarity matrix training **based on similar repetitors**

In [3]:
from sqlalchemy import select
from sqlalchemy.orm import joinedload
from models import boulder
from models.area import Area
from models.ascent import Ascent
from models.boulder import Boulder
from models.crag import Crag
from models.grade import Grade

area_slug = "ticino"

boulders = (
    session.execute(
        select(Boulder)
        .join(Boulder.crag)
        .join(Crag.area)
        .join(Boulder.grade)
        .options(joinedload(Boulder.grade))
        .where(Area.slug == area_slug)
        .order_by(Grade.correspondence, Boulder.name)
    )
    .scalars()
    .all()
)

for index, boulder in enumerate(boulders):
    boulder.similarity_matrix_id = index
    session.add(boulder)
session.commit()

### Database Query

In [4]:
ascents = session.execute(
    select(Ascent.user_id, Boulder.similarity_matrix_id)
    .join(Ascent.boulder)
    .join(Boulder.crag)
    .join(Crag.area)
    .where(Area.slug == area_slug)
).all()
ascents_df = pd.DataFrame(data=ascents, columns=["user_id", "id"])
ascents_df

,user_id,id
0,126,0
1,327,1
2,334,3
3,108,3
4,52,4
...,...,...
74750,1777,6231
74751,3187,7098
74752,765,5759
74753,436,5759


### boulder_user matrix (Pivot table)

In [5]:
boulder_user_matrix = ascents_df.pivot_table(
    index="id",
    columns="user_id",
    aggfunc="size",
    fill_value=0,
    dropna=True,
)
# boulder_user_matrix = boulder_user_matrix[boulder_user_matrix.index < 20]
boulder_ids = boulder_user_matrix.index

In [6]:
display(boulder_user_matrix)

user_id,1,2,3,4,5,6,7,8,9,10,...,4218,4219,4220,4221,4222,4223,4224,4225,4226,4227
id,,,,,,,,,,,,,,,,,,,,,
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
5,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7231,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
7232,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
7233,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


### Conversion to sparse matrix

In [7]:
from scipy.sparse import coo_matrix

boulder_user_matrix = coo_matrix(boulder_user_matrix)
print(boulder_user_matrix)

<COOrdinate sparse matrix of dtype 'int64'
	with 73551 stored elements and shape (5749, 4227)>
  Coords	Values
  (0, 125)	1
  (1, 326)	1
  (2, 107)	1
  (2, 333)	1
  (3, 51)	1
  (3, 62)	1
  (3, 107)	1
  (4, 770)	1
  (5, 742)	2
  (6, 770)	1
  (7, 753)	1
  (8, 328)	1
  (9, 331)	1
  (10, 770)	1
  (11, 122)	1
  (11, 741)	1
  (12, 741)	1
  (13, 742)	1
  (14, 770)	1
  (15, 759)	1
  (16, 53)	1
  (16, 509)	1
  (17, 742)	1
  (18, 74)	1
  (18, 119)	1
  :	:
  (5740, 3358)	1
  (5740, 4066)	1
  (5740, 4125)	1
  (5741, 1611)	1
  (5741, 1719)	1
  (5741, 3826)	1
  (5741, 3959)	1
  (5741, 4088)	1
  (5741, 4139)	1
  (5742, 1719)	1
  (5742, 3100)	1
  (5742, 3826)	1
  (5742, 3834)	1
  (5742, 3959)	1
  (5743, 1719)	1
  (5743, 2930)	1
  (5743, 3826)	1
  (5744, 1719)	1
  (5744, 2930)	1
  (5744, 3635)	1
  (5745, 3959)	1
  (5746, 3959)	1
  (5747, 2635)	1
  (5747, 4125)	1
  (5748, 4088)	1


### Ascents similarity training

In [8]:
def jaccard_pairwise_similarity(X):
    """Compute the Jaccard similarity between pairs of boulders based on user ascents.
    Args:
        X (scipy.sparse.csr_matrix): Sparse matrix of shape (n_boulders, n_users)
            where each entry indicates whether a user has ascended a boulder.
    Returns:
        scipy.sparse.coo_matrix: Sparse matrix of shape (n_boulders, n_boulders)
            containing the Jaccard similarity between each pair of boulders.
    """
    # CSR matrix storing the number of shared ascents for each pair of
    # boulders sharing at least one ascent

    intersection = X @ X.T

    # 1D array storing the total number of ascent for each boulder
    row_sums = np.asarray(X.sum(axis=1)).ravel()

    # intersection decomposition for calculation on 1D arrays
    rows, cols = intersection.nonzero()
    intersection_data = intersection.data

    union = row_sums[rows] + row_sums[cols] - intersection_data

    # Avoid division by zero
    mask = union > 0
    jaccard = np.zeros_like(intersection_data, dtype=np.float32)
    jaccard[mask] = intersection_data[mask] / union[mask]

    # Index remapping based on the boulder ids
    new_rows = boulder_ids[rows]
    new_cols = boulder_ids[cols]

    return coo_matrix(
        (jaccard, (new_rows, new_cols)),
        dtype=np.float32,
    )

similarity_ascents = jaccard_pairwise_similarity(boulder_user_matrix)

sparsity = 1.0 - similarity_ascents.nnz / (
    similarity_ascents.shape[0] * similarity_ascents.shape[1]
)

print(f"Sparsity: {sparsity:.2%}")

Sparsity: 96.23%


In [9]:
# similarity_ascents_df = pd.DataFrame(similarity_ascents.toarray())
# display(similarity_ascents_df)

## Similarity matrix training **based on similar features**

### Database query and dataframe creation

In [10]:
# Data extraction
boulders_list = [
    {
        "id": boulder.similarity_matrix_id,
        "grade": boulder.grade.correspondence,
    }
    for boulder in boulders
]

# Dataframe setup
boulders_df = pd.DataFrame(boulders_list)
# boulders_df = boulders_df[boulders_df.id < 20]
boulder_ids = boulders_df.id
boulders_df.head()


,id,grade
0,0,12
1,1,12
2,2,12
3,3,12
4,4,12


### Grade

#### Fuzzy one-hot grade vector

In [11]:
max_grade = boulders_df.grade.max()
min_grade = boulders_df.grade.min()
grade_df = pd.get_dummies(boulders_df.grade, dtype=np.float32)
grade_df

,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30
0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7231,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
7232,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
7233,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
7234,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0


In [12]:
def grade_update(row, min_grade, max_grade):
    grade_index = row.idxmax()

    if grade_index == 0:
        return row

    values = np.array([0.5, 0.75, 0.75, 0.5], dtype=np.float32)
    offsets = np.array([-2, -1, 1, 2])

    for offset, value in zip(offsets, values):
        current_column = grade_index + offset
        if min_grade < current_column <= max_grade:
            row[current_column] = value
    return row


grade_df = grade_df.apply(
    lambda row: grade_update(row, min_grade=min_grade, max_grade=max_grade),
    axis=1,
)
grade_df.fillna(0, inplace=True)
display(grade_df)

,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30
0,1.0,0.75,0.5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.00,0.00
1,1.0,0.75,0.5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.00,0.00
2,1.0,0.75,0.5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.00,0.00
3,1.0,0.75,0.5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.00,0.00
4,1.0,0.75,0.5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.00,0.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7231,0.0,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.5,0.75,1.00,0.75
7232,0.0,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.5,0.75,1.00,0.75
7233,0.0,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.5,0.75,1.00,0.75
7234,0.0,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.5,0.75,1.00,0.75


#### Conversion to sparse matrix

In [13]:
grade = coo_matrix(grade_df)

#### Grade similarity training

In [14]:
from time import perf_counter
from sklearn.metrics.pairwise import cosine_similarity

# Cosine training
start = perf_counter()
similarity_grade = cosine_similarity(grade)
end = perf_counter()
print(f"Cosine calculation time: {end - start:.4f}")


# Re-indexing to match database
start = perf_counter()
similarity_grade = coo_matrix(similarity_grade)
end = perf_counter()
print(f"COO conversion time: {end - start:.4f}")

# Sparcity
sparsity = 1 - similarity_grade.nnz / (
    similarity_grade.shape[0] * similarity_grade.shape[1]
)
print(f"Sparcity: {sparsity:.2f}")

# similarity_grade_df = pd.DataFrame(similarity_grade.toarray())
# display(similarity_grade_df)

Cosine calculation time: 0.0775
COO conversion time: 0.5405
Sparcity: 0.45


In [15]:
print(similarity_grade)

<COOrdinate sparse matrix of dtype 'float32'
	with 28776744 stored elements and shape (7236, 7236)>
  Coords	Values
  (0, 0)	1.0
  (0, 1)	1.0
  (0, 2)	1.0
  (0, 3)	1.0
  (0, 4)	1.0
  (0, 5)	1.0
  (0, 6)	1.0
  (0, 7)	1.0
  (0, 8)	1.0
  (0, 9)	1.0
  (0, 10)	1.0
  (0, 11)	1.0
  (0, 12)	1.0
  (0, 13)	1.0
  (0, 14)	1.0
  (0, 15)	1.0
  (0, 16)	1.0
  (0, 17)	1.0
  (0, 18)	1.0
  (0, 19)	1.0
  (0, 20)	1.0
  (0, 21)	1.0
  (0, 22)	1.0
  (0, 23)	1.0
  (0, 24)	1.0
  :	:
  (7235, 7211)	0.7163352966308594
  (7235, 7212)	0.7163352966308594
  (7235, 7213)	0.7163352966308594
  (7235, 7214)	0.7163352966308594
  (7235, 7215)	0.7163352966308594
  (7235, 7216)	0.7163352966308594
  (7235, 7217)	0.7163352966308594
  (7235, 7218)	0.7163352966308594
  (7235, 7219)	0.7163352966308594
  (7235, 7220)	0.7163352966308594
  (7235, 7221)	0.7163352966308594
  (7235, 7222)	0.7163352966308594
  (7235, 7223)	0.7163352966308594
  (7235, 7224)	0.7163352966308594
  (7235, 7225)	0.7163352966308594
  (7235, 7226)	0.90371280908

## Coherence check

In [16]:
print(type(similarity_ascents))
print(type(similarity_grade))

print(similarity_ascents.shape)
print(similarity_grade.shape)

print(similarity_ascents.nnz)
print(similarity_grade.nnz)

print(similarity_ascents.dtype)
print(similarity_grade.dtype)

<class 'scipy.sparse._coo.coo_matrix'>
<class 'scipy.sparse._coo.coo_matrix'>
(7236, 7236)
(7236, 7236)
1975265
28776744
float32
float32


## Data cleaning
Removing all values in similarity_grade < 0.5 (grade difference >= 2)  
Removing of all non zero values in similarity_ascents and similarity_style where similarity_grade == 0

In [17]:
similarity_grade_cleaned = similarity_grade.copy()
similarity_grade_cleaned.data[similarity_grade_cleaned.data < 0.5] = 0
similarity_grade_cleaned.eliminate_zeros()

In [18]:
similarity_grade_cleaned = similarity_grade_cleaned.tocsr()

In [19]:
from scipy.sparse import csr_matrix

def matrix_cleaning(cleaning_matrix: csr_matrix, matrix_to_clean: coo_matrix):
    """Remove all values in matrix_to_clean that are not indexed in cleaning_matrix
    
    :parameters:
    cleaning_matrix: CSR matrix that serves as reference for data existence
    matrix_to_clean: COO matrix from which some indexes are removed"""
    
    # Boolean mask creation (1D array) - Fancy indexing converted to boolean
    # Check if cleaning_matrix contains the indexes of matrix_to_clean
    mask = cleaning_matrix[matrix_to_clean.row, matrix_to_clean.col].A1 != 0

    # Fancy indexing to remove values from matrix_to_clean that are equal to 0 
    # in cleaning_matrix
    new_rows = matrix_to_clean.row[mask]
    new_cols = matrix_to_clean.col[mask]
    new_data = matrix_to_clean.data[mask]

    return csr_matrix(
        (new_data, (new_rows, new_cols)), shape=matrix_to_clean.shape
    )

similarity_ascents_cleaned = matrix_cleaning(similarity_grade_cleaned, similarity_ascents)

In [20]:
print(
    1
    - similarity_ascents.nnz
    / (similarity_ascents.shape[0] * similarity_ascents.shape[1])
)
print(
    1
    - similarity_ascents_cleaned.nnz
    / (
        similarity_ascents_cleaned.shape[0]
        * similarity_ascents_cleaned.shape[1]
    )
)
print(
    1
    - similarity_grade.nnz
    / (similarity_grade.shape[0] * similarity_grade.shape[1])
)
print(
    1
    - similarity_grade_cleaned.nnz
    / (similarity_grade_cleaned.shape[0] * similarity_grade_cleaned.shape[1])
)

0.962275086547485
0.9795557063585701
0.4504027678082776
0.6617062482562923


## Matrix saving

In [21]:
from scipy.sparse import save_npz

# save_npz("../../similarity_ascent.npz", similarity_ascents_cleaned)
# save_npz("../../similarity_grade.npz", similarity_grade_cleaned)

## Recommendation example


In [22]:
def recommend_boulders(input_boulders, top_n=5, alpha=0.7, beta=0.3):

    ascents = similarity_ascents_cleaned[:, input_boulders].sum(axis=1).A1
    grade = similarity_grade_cleaned[:, input_boulders].sum(axis=1).A1

    ascents[input_boulders] = 0
    grade[input_boulders] = 0

    sim_scores = alpha * ascents + beta * grade

    best_boulders = np.argsort(-sim_scores)[:top_n]

    return best_boulders.tolist()


recommendations = recommend_boulders([6741], top_n=20)


boulders = session.execute(
    select(Boulder.name, Boulder.similarity_matrix_id).filter(
        Boulder.similarity_matrix_id.in_(recommendations)
    )
).all()

boulder_dict = {
    boulder_id: boulder_name for boulder_name, boulder_id in boulders
}
boulder_names = [(boulder_dict[boulder_id], boulder_id) for boulder_id in recommendations]
display(boulder_names)

[('Tortillas revolution', 6116),
 ("Murphy's low", 6388),
 ('Schlonziges Wiener Schmankerl', 6743),
 ('Entwash', 6584),
 ('backgammon sit start', 6815),
 ('Backdoor', 6540),
 ('Autant en emporte le vent', 6539),
 ('Auf leisen Pfoten', 6538),
 ('Attrazione', 6537),
 ('Atlantide', 6536),
 ('Arête with the Pocket', 6535),
 ('Areta with the poket', 6534),
 ('3°Cane', 6527),
 ('3° Cane', 6526),
 ('3 cane', 6525),
 ('3 Degree Cane', 6524),
 ('3 Crane', 6523),
 ('3 Cane', 6522),
 ('21 power', 6521),
 ('Trigonometry', 6800)]